In [7]:
from pathlib import Path
import pandas as pd

pasta_projeto = Path.cwd()
arquivo_2012 = pasta_projeto / "Tabela5-sem_emprego_2012.csv"
arquivo_2026 = pasta_projeto / "Tabela5-sem_emprego_2026.csv"
arquivo_cuidados = pasta_projeto / "Tabela 1.1.1.xls"

for arquivo in [arquivo_2012, arquivo_2026, arquivo_cuidados]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo.name}")

print("Arquivos encontrados com sucesso.")

Arquivos encontrados com sucesso.


In [8]:
t2012 = pd.read_csv(
    arquivo_2012,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

t2026 = pd.read_csv(
    arquivo_2026,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

print("Tabela 2012:")
print(t2012.head().to_string(index=False))
print("\nTabela 2026:")
print(t2026.head().to_string(index=False))

Tabela 2012:
Sigla  Código   Estado  Desocupados - homens (2012 T1)  Desocupados - mulheres (2012 T1)
   AC      12     Acre                            45.3                              54.7
   AL      27  Alagoas                            44.7                              55.3
   AM      13 Amazonas                            47.5                              52.5
   AP      16    Amapá                            48.1                              51.9
   BA      29    Bahia                            43.8                              56.2

Tabela 2026:
Sigla  Código   Estado  Desocupados - homens (2026 T1)  Desocupados - mulheres (2026 T1)
   AC      12     Acre                            53.2                              46.8
   AL      27  Alagoas                            50.7                              49.3
   AM      13 Amazonas                            46.8                              53.2
   AP      16    Amapá                            51.1                             

In [9]:
comp = t2012.merge(
    t2026,
    on=["Sigla", "Código", "Estado"],
    how="inner",
)

comp = comp.rename(columns={
    "Desocupados - homens (2012 T1)": "desocupados_homens_2012",
    "Desocupados - mulheres (2012 T1)": "desocupados_mulheres_2012",
    "Desocupados - homens (2026 T1)": "desocupados_homens_2026",
    "Desocupados - mulheres (2026 T1)": "desocupados_mulheres_2026",
})

comp["variacao_homens_pp"] = (
    comp["desocupados_homens_2026"] - comp["desocupados_homens_2012"]
).round(1)
comp["variacao_mulheres_pp"] = (
    comp["desocupados_mulheres_2026"] - comp["desocupados_mulheres_2012"]
).round(1)

print(f"UFs após o primeiro merge: {len(comp)}")
print(comp.head().to_string(index=False))

UFs após o primeiro merge: 27
Sigla  Código   Estado  desocupados_homens_2012  desocupados_mulheres_2012  desocupados_homens_2026  desocupados_mulheres_2026  variacao_homens_pp  variacao_mulheres_pp
   AC      12     Acre                     45.3                       54.7                     53.2                       46.8                 7.9                  -7.9
   AL      27  Alagoas                     44.7                       55.3                     50.7                       49.3                 6.0                  -6.0
   AM      13 Amazonas                     47.5                       52.5                     46.8                       53.2                -0.7                   0.7
   AP      16    Amapá                     48.1                       51.9                     51.1                       48.9                 3.0                  -3.0
   BA      29    Bahia                     43.8                       56.2                     44.7                       55.

In [10]:
bruto = pd.read_excel(
    arquivo_cuidados,
    sheet_name="2022",
    engine="xlrd",
    header=None,
)

cuidados = bruto.iloc[8:41].copy()
cuidados.columns = [
    "Estado",
    "horas_total",
    "horas_total_branca",
    "horas_total_preta_parda",
    "horas_homem_branca",
    "horas_homem_preta_parda",
    "horas_mulher_branca",
    "horas_mulher_preta_parda",
]
cuidados = cuidados.reset_index(drop=True)

for coluna in cuidados.columns[1:]:
    cuidados[coluna] = pd.to_numeric(cuidados[coluna], errors="coerce")

regioes = ["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
cuidados = cuidados[~cuidados["Estado"].isin(regioes)].copy()

cuidados["horas_homens_media"] = (
    cuidados["horas_homem_branca"] + cuidados["horas_homem_preta_parda"]
) / 2
cuidados["horas_mulheres_media"] = (
    cuidados["horas_mulher_branca"] + cuidados["horas_mulher_preta_parda"]
) / 2
cuidados["diferenca_horas_mulheres_homens"] = (
    cuidados["horas_mulheres_media"] - cuidados["horas_homens_media"]
).round(2)

print(f"UFs na Tabela 1.1.1: {len(cuidados)}")
print(cuidados.head().to_string(index=False))

UFs na Tabela 1.1.1: 27
  Estado  horas_total  horas_total_branca  horas_total_preta_parda  horas_homem_branca  horas_homem_preta_parda  horas_mulher_branca  horas_mulher_preta_parda  horas_homens_media  horas_mulheres_media  diferenca_horas_mulheres_homens
Rondônia    17.014162           16.727733                17.101523           12.111127                12.544080            20.412887                 21.061410           12.327604             20.737149                             8.41
    Acre    13.606903           13.806929                13.541399           10.796120                 9.686983            16.130352                 16.710754           10.241551             16.420553                             6.18
Amazonas    16.807290           15.987545                16.995493           12.233043                12.822584            18.764969                 20.716986           12.527813             19.740977                             7.21
 Roraima    13.639257           13.65175

In [5]:
cruzado = comp.merge(
    cuidados,
    on="Estado",
    how="inner",
)

cruzado = cruzado.sort_values("Código").reset_index(drop=True)

if len(cruzado) != 27:
    raise ValueError(f"O cruzamento deveria ter 27 UFs, mas possui {len(cruzado)}.")

print(f"UFs na tabela final: {len(cruzado)}")
print(cruzado.head().to_string(index=False))

UFs na tabela final: 27
Sigla  Código   Estado  desocupados_homens_2012  desocupados_mulheres_2012  desocupados_homens_2026  desocupados_mulheres_2026  variacao_homens_pp  variacao_mulheres_pp  horas_total  horas_total_branca  horas_total_preta_parda  horas_homem_branca  horas_homem_preta_parda  horas_mulher_branca  horas_mulher_preta_parda  horas_homens_media  horas_mulheres_media  diferenca_horas_mulheres_homens
   RO      11 Rondônia                     43.4                       56.6                     33.0                       67.0               -10.4                  10.4    17.014162           16.727733                17.101523           12.111127                12.544080            20.412887                 21.061410           12.327604             20.737149                             8.41
   AC      12     Acre                     45.3                       54.7                     53.2                       46.8                 7.9                  -7.9    13.606903       

In [6]:
arquivo_saida = pasta_projeto / "geral.csv"

cruzado.to_csv(
    arquivo_saida,
    sep=";",
    decimal=",",
    index=False,
    encoding="utf-8-sig",
)

print(f"Arquivo salvo: {arquivo_saida.name}")

Arquivo salvo: geral.csv
